# OTEL Metrics — Silver Pipeline

Spark Declarative Pipeline that creates silver tables from raw OTEL metrics data.

| Table | Type | Description |
| --- | --- | --- |
| `node_compute` | Streaming | Node CPU & memory metrics (long format) |
| `node_filesystem` | Materialized View | Node filesystem with pivoted available/capacity/usage |
| `node_network` | Streaming | Node network IO with interface and direction |
| `pod_compute` | Streaming | Pod CPU & memory metrics with full K8s hierarchy |
| `pod_filesystem` | Materialized View | Pod filesystem with pivoted available/capacity/usage |
| `pod_network` | Streaming | Pod network IO with interface and direction |

In [ ]:
import dlt
from pyspark.sql import functions as F

source_catalog = spark.conf.get("source_catalog")
source_schema = spark.conf.get("source_schema")
table_prefix = spark.conf.get("table_prefix")
source_table = f"{source_catalog}.{source_schema}.{table_prefix}_otel_metrics"

In [ ]:
# Bronze v2: top-level `time` (TIMESTAMP) and `time_unix_nano` (LONG) replace
# the per-struct *_time_unix_nano fields that lived inside gauge/sum/etc. in v1.
# All attribute maps are now VARIANT — accessed via `:['key']::type`.

In [ ]:
@dlt.table(
    comment="Node-level CPU and memory metrics — one row per metric per timestamp",
)
def node_compute():
    return (
        spark.readStream.table(source_table)
        .filter(
            (F.col("name").like("k8s.node.cpu.%")) | (F.col("name").like("k8s.node.memory.%"))
        )
        .filter(F.col("metric_type").isin("gauge", "sum"))
        .selectExpr(
            "time AS timestamp",
            "resource.attributes:['k8s.node.name']::string AS node",
            "name AS metric_name",
            "REPLACE(REPLACE(name, 'k8s.node.cpu.', ''), 'k8s.node.memory.', '') AS metric_short_name",
            "CASE WHEN name LIKE '%.cpu.%' THEN 'cpu' ELSE 'memory' END AS metric_domain",
            "metric_type",
            "CASE WHEN metric_type = 'sum' THEN sum.is_monotonic END AS is_monotonic",
            """CASE
                WHEN metric_type = 'gauge' THEN gauge.value
                WHEN metric_type = 'sum'   THEN sum.value
            END AS value""",
            "unit",
        )
    )

In [ ]:
@dlt.table(
    comment="Node-level filesystem metrics — pivoted with available/capacity/usage per timestamp",
)
def node_filesystem():
    return (
        spark.read.table(source_table)
        .filter(
            (F.col("name").like("k8s.node.filesystem.%")) & (F.col("metric_type") == "gauge")
        )
        .selectExpr(
            "time AS timestamp",
            "resource.attributes:['k8s.node.name']::string AS node",
            "name",
            "gauge.value AS value",
        )
        .groupBy("timestamp", "node")
        .agg(
            F.max(F.when(F.col("name") == "k8s.node.filesystem.available", F.col("value"))).alias("available_bytes"),
            F.max(F.when(F.col("name") == "k8s.node.filesystem.capacity", F.col("value"))).alias("capacity_bytes"),
            F.max(F.when(F.col("name") == "k8s.node.filesystem.usage", F.col("value"))).alias("usage_bytes"),
        )
        .withColumn(
            "usage_pct",
            F.round(F.col("usage_bytes") * 100.0 / F.when(F.col("capacity_bytes") != 0, F.col("capacity_bytes")), 2),
        )
    )

In [ ]:
@dlt.table(
    comment="Node-level network IO and error metrics with interface and direction",
)
def node_network():
    return (
        spark.readStream.table(source_table)
        .filter(
            (F.col("name").like("k8s.node.network.%")) & (F.col("metric_type") == "sum")
        )
        .selectExpr(
            "time AS timestamp",
            "resource.attributes:['k8s.node.name']::string AS node",
            "name AS metric_name",
            "REPLACE(name, 'k8s.node.network.', '') AS metric_short_name",
            "sum.attributes:['interface']::string AS interface",
            "sum.attributes:['direction']::string AS direction",
            "sum.is_monotonic",
            "sum.value AS value",
            "unit",
        )
    )

In [ ]:
@dlt.table(
    comment="Pod-level CPU and memory metrics with full K8s resource hierarchy",
)
def pod_compute():
    return (
        spark.readStream.table(source_table)
        .filter(
            (F.col("name").like("k8s.pod.cpu.%")) | (F.col("name").like("k8s.pod.memory.%"))
        )
        .filter(F.col("metric_type").isin("gauge", "sum"))
        .selectExpr(
            "time AS timestamp",
            "resource.attributes:['k8s.node.name']::string AS node",
            "resource.attributes:['k8s.namespace.name']::string AS namespace",
            "resource.attributes:['k8s.deployment.name']::string AS deployment",
            "resource.attributes:['k8s.pod.name']::string AS pod",
            "resource.attributes:['k8s.pod.uid']::string AS pod_uid",
            "resource.attributes:['k8s.container.name']::string AS container",
            "service_name AS service",
            "resource.attributes:['service.namespace']::string AS service_namespace",
            "name AS metric_name",
            "REPLACE(REPLACE(name, 'k8s.pod.cpu.', ''), 'k8s.pod.memory.', '') AS metric_short_name",
            "CASE WHEN name LIKE '%.cpu.%' THEN 'cpu' ELSE 'memory' END AS metric_domain",
            "metric_type",
            "CASE WHEN metric_type = 'sum' THEN sum.is_monotonic END AS is_monotonic",
            """CASE
                WHEN metric_type = 'gauge' THEN gauge.value
                WHEN metric_type = 'sum'   THEN sum.value
            END AS value""",
            "unit",
        )
    )

In [ ]:
@dlt.table(
    comment="Pod-level filesystem metrics — pivoted with available/capacity/usage per timestamp",
)
def pod_filesystem():
    return (
        spark.read.table(source_table)
        .filter(
            (F.col("name").like("k8s.pod.filesystem.%")) & (F.col("metric_type") == "gauge")
        )
        .selectExpr(
            "time AS timestamp",
            "resource.attributes:['k8s.node.name']::string AS node",
            "resource.attributes:['k8s.namespace.name']::string AS namespace",
            "resource.attributes:['k8s.deployment.name']::string AS deployment",
            "resource.attributes:['k8s.pod.name']::string AS pod",
            "resource.attributes:['k8s.pod.uid']::string AS pod_uid",
            "service_name AS service",
            "name",
            "gauge.value AS value",
        )
        .groupBy("timestamp", "node", "namespace", "deployment", "pod", "pod_uid", "service")
        .agg(
            F.max(F.when(F.col("name") == "k8s.pod.filesystem.available", F.col("value"))).alias("available_bytes"),
            F.max(F.when(F.col("name") == "k8s.pod.filesystem.capacity", F.col("value"))).alias("capacity_bytes"),
            F.max(F.when(F.col("name") == "k8s.pod.filesystem.usage", F.col("value"))).alias("usage_bytes"),
        )
        .withColumn(
            "usage_pct",
            F.round(F.col("usage_bytes") * 100.0 / F.when(F.col("capacity_bytes") != 0, F.col("capacity_bytes")), 2),
        )
    )

In [ ]:
@dlt.table(
    comment="Pod-level network IO and error metrics with interface and direction",
)
def pod_network():
    return (
        spark.readStream.table(source_table)
        .filter(
            (F.col("name").like("k8s.pod.network.%")) & (F.col("metric_type") == "sum")
        )
        .selectExpr(
            "time AS timestamp",
            "resource.attributes:['k8s.node.name']::string AS node",
            "resource.attributes:['k8s.namespace.name']::string AS namespace",
            "resource.attributes:['k8s.deployment.name']::string AS deployment",
            "resource.attributes:['k8s.pod.name']::string AS pod",
            "resource.attributes:['k8s.pod.uid']::string AS pod_uid",
            "service_name AS service",
            "name AS metric_name",
            "REPLACE(name, 'k8s.pod.network.', '') AS metric_short_name",
            "sum.attributes:['interface']::string AS interface",
            "sum.attributes:['direction']::string AS direction",
            "sum.is_monotonic",
            "sum.value AS value",
            "unit",
        )
    )